In [1]:
import yaml
import torch
import matplotlib.pyplot as plt
import os
import glob
import numpy as np
import argparse
import plotly.graph_objects as go
import monai.losses
from monai.utils import set_determinism
from monai.metrics import DiceMetric
from monai.data import CacheDataset, Dataset, decollate_batch, ThreadDataLoader
from monai.config import print_config
from monai.apps import CrossValidation
from aim.pytorch import track_gradients_dists, track_params_dists
from abc import ABC, abstractmethod
from collections import defaultdict
from monai_train.transformer import mtrain_transforms, kfold_transforms
from monai_train.earlystop import EarlyStopping
from monai_train.progress_bar import MyProgressBar


class CVDataset(ABC, CacheDataset):
    """
    Base class to generate cross validation datasets.

    """
    
    def __init__(
        self,
        data,
        transform,
        cache_rate=1.0,
        num_workers=4,
    ) -> None:
        data = self._split_datalist(datalist=data)
        CacheDataset.__init__(
            self, data, transform, cache_rate=cache_rate, num_workers=num_workers
        )

    @abstractmethod
    def _split_datalist(self, datalist):
        raise NotImplementedError(f"Subclass {self.__class__.__name__} must implement this method.")

class CustomDataset(Dataset):
    def __init__(self, image_label_pairs):
        self.image_label_pairs = image_label_pairs

    def __len__(self):
        return len(self.image_label_pairs)

    def __getitem__(self, idx):
        sample = self.image_label_pairs[idx]
        image_path = sample['image']
        label_path = sample['label']
        # Load the image and label (modify this if using a specific loader, e.g., nibabel for .nii.gz)
        image = self.load_image(image_path)
        label = self.load_image(label_path)
        return image, label

    def load_image(self, path):
        # Implement image loading here (e.g., nibabel for .nii.gz files)
        # For now, this is a placeholder function
        return path
    
class CustomDataloader:
    def __init__(self, image_label_pairs, split_ratio=0.8, batch_size=1, seed=0):
        self.image_label_pairs = image_label_pairs
        self.split_ratio = split_ratio
        self.batch_size = batch_size
        self.seed = seed

        # Group images
        self.grouped_images = self.group_images()

        # Split into training and validation sets
        self.training_set, self.validation_set = self.split_groups()

        # Create datasets
        self.train_dataset = CustomDataset(self.training_set)
        self.val_dataset = CustomDataset(self.validation_set)

    def group_images(self):
        """Group images by a common identifier (assumed to be part of the file name).
        Requested by Olivia & Tomer (Umich)"""
        groups = defaultdict(list)

        for pair in self.image_label_pairs:
            # Assuming the group identifier is part of the filename after "cropped filtered"
            group_id = pair['image'].split('_', 3)[0]  # Adjust as needed for your filenames
            groups[group_id].append(pair)
        return list(groups.values())

    def split_groups(self):
        """Split the groups into training and validation sets using torch.utils.data.random_split with a seed."""
        total_groups = len(self.grouped_images)
        split_index = int(total_groups * self.split_ratio)

        # Set up a generator for reproducibility
        generator = torch.Generator().manual_seed(self.seed)

        # Perform random split
        train_groups, val_groups = torch.utils.data.random_split(self.grouped_images, [split_index, total_groups - split_index], generator=generator)

        # Flatten the groups into individual image-label pairs
        training_set = [item for group in train_groups for item in group]
        validation_set = [item for group in val_groups for item in group]

        return training_set, validation_set


def get_data_dict(data_dir: str) -> list(): # type: ignore
    """
    Return data list for kfold preparation

    Args:
        data_dir (str): Path to the directory containing the data.
    
    Returns:
        list: A list containing all training & validation data.

    """
    try:
        # Check if data_dir exists
        if not os.path.isdir(data_dir):
            raise Exception(f"The directory '{data_dir}' does not exist.")            

        # Check for the presence of required folders
        required_folders = ["imagesTr", "labelsTr", "imagesTs"]
        for folder in required_folders:
            if not os.path.isdir(os.path.join(data_dir, folder)):
                raise Exception(f"The directory '{folder}' does not exist in '{data_dir}'.")

        # Check if each image in imagesTr has a corresponding label in labelsTr
        imagesTr_files = os.listdir(os.path.join(data_dir, "imagesTr"))
        labelsTr_files = os.listdir(os.path.join(data_dir, "labelsTr"))
        for image_file in imagesTr_files:
            if image_file not in labelsTr_files:
                raise Exception(f"No matching label found for the image '{image_file}' in 'labelsTr' folder.")

        print("All conditions met.")
    except Exception as e:
        print("Error:", e)

    # Get list of training images, and their labels
    train_images = sorted(glob.glob(os.path.join(data_dir, "imagesTr", "*.nii.gz")))
    train_labels = sorted(glob.glob(os.path.join(data_dir, "labelsTr", "*.nii.gz")))
    data_dicts = [{"image": image_name, "label": label_name} for image_name, label_name in zip(train_images, train_labels)]

    return data_dicts
    
def load_data(data_dir: str, split: float, cache_rate:float, workers: int, batch_size:int, image_size:tuple, roi_size:tuple, seed:int, group_similar:bool) -> list(): # type: ignore
    """
    Load data for training and validation.

    Args:
        data_dir (str): Path to the directory containing the data.
        split (float): Percentage of data to be used for training (0 to 1).
        train_transforms: Transformations to be applied to training data.
        val_transforms: Transformations to be applied to validation data.
        cache_rate (float): Percentage of data to cache.
        workers (int): Number of worker processes for data loading.
        batch_size (int): Batch size for data loading.

    Returns:
        list: A list containing the training DataLoader, validation DataLoader, and training CacheDataset.
    
    Raises:
        Exception: If the data directory or required folders do not exist, or if there are missing labels for images.

    """
    try:
        # Check if data_dir exists
        if not os.path.isdir(data_dir):
            raise Exception(f"The directory '{data_dir}' does not exist.")            

        # Check for the presence of required folders
        required_folders = ["imagesTr", "labelsTr", "imagesTs"]
        for folder in required_folders:
            if not os.path.isdir(os.path.join(data_dir, folder)):
                raise Exception(f"The directory '{folder}' does not exist in '{data_dir}'.")

        # Check if each image in imagesTr has a corresponding label in labelsTr
        imagesTr_files = os.listdir(os.path.join(data_dir, "imagesTr"))
        labelsTr_files = os.listdir(os.path.join(data_dir, "labelsTr"))
        for image_file in imagesTr_files:
            if image_file not in labelsTr_files:
                raise Exception(f"No matching label found for the image '{image_file}' in 'labelsTr' folder.")

        print("All conditions met.")
    except Exception as e:
        print("Error:", e)

    # Get list of all training images, and their labels
    train_images = sorted(glob.glob(os.path.join(data_dir, "imagesTr", "*.nii.gz")))
    train_labels = sorted(glob.glob(os.path.join(data_dir, "labelsTr", "*.nii.gz")))
    data_dicts = [{"image": image_name, "label": label_name} for image_name, label_name in zip(train_images, train_labels)]

    if group_similar:
        custom_dataloader = CustomDataloader(data_dicts, split_ratio=split, batch_size=batch_size, seed=seed)
        # Flatten the lists (since we grouped them earlier)
        training_set, validation_set = custom_dataloader.split_groups()
    else:
        # Split training data into training and validation
        train_size = int(split * len(data_dicts))
        val_size = len(data_dicts) - train_size
        training_set, validation_set = torch.utils.data.random_split(data_dicts, [train_size, val_size])


    print("***********", end="\n\n")
    for x in training_set:
        print(x)


    # get transformations
    train_transforms, val_transforms = mtrain_transforms(image_size, roi_size=roi_size)

    # Create training dataloader
    train_ds = CacheDataset(data=training_set, transform=train_transforms, cache_rate=cache_rate, num_workers=workers)
    train_loader = ThreadDataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    # Create validation dataloader
    val_ds = CacheDataset(data=validation_set, transform=val_transforms, cache_rate=cache_rate, num_workers=workers)
    val_loader = ThreadDataLoader(val_ds, batch_size=1, num_workers=0)

    return [train_loader, val_loader, train_ds, val_ds]

def create_parser():
    parser = argparse.ArgumentParser()
    g = parser.add_argument_group('MONAI Targets')
    g.add_argument(
        '-model', '--model',
        dest='model_file',
        type=str, help="path to model configuration file (yaml)")
    g.add_argument(
        '-data', '--data',
        dest='data_dir',
        type=str, help="path to training data (dir)")
    g.add_argument(
        '-output', '--output',
        dest='output_dir',
        type=str, help="path to output folder (dir)")
    g.add_argument(
        '-split', '--split',
        dest='split_percentage',
        type=float,
        default=0.8, help="fraction to split training data into training/validation pair")
    g.add_argument(
        '-lr', '--lr',
        dest='learning_rate',
        default=0.0001,
        type=float, help="training optimizer's learning rate (float)")
    g.add_argument(
        '-epochs', '--epochs',
        dest='epochs',
        default=100,
        type=int, help="total number of epoch's per training")
    g.add_argument(
        '-batch', '--batch',
        dest='batch_size',
        default=1,
        type=int, help="training batch size")
    g.add_argument(
        '-transfer', '--transfer',
        dest='transfer_learning',
        type=str, help="path to trained model file (/path/to/file/*.pth) for transfer learning (path)")
    g.add_argument(
        '-kfold', '--kfold',
        dest='kfold',
        type=int, help="total number of K-fold sessions, enable with any value >= 1")
    g.add_argument(
        '-savemodel', '--savemodel',
        dest='savemodel',
        type=bool, help="save final trained model with the best mean-dice score")
    g.add_argument(
        '-seed', '--seed',
        dest='seed',
        type=int, 
        default=0)
    g.add_argument(
        '-group-similar', '--group-similar',
        dest='group_similar',
        action="count", help="group together similar images. The training/validation split will not split grouped images. Images are to be grouped by a shared unique ID")
    g.add_argument(
        '-show-config', '--show-config',
        dest='show_config',
        action="count")
    g.add_argument(
        '-early-stopping', '--early-stopping',
        dest='early_stopping',
        action="count")
    g.add_argument(
        '-min-epochs', '--min-epochs',
        dest='min_epochs',
        type=int, help="Minimum number of epochs before checking for early-stopping criteria")
    g.add_argument(
        '-patience', '--patience',
        dest='patience',
        type=int, help="Number of epochs to wait if there is no significant improvement")
    g.add_argument(
        '-threshold', '--threshold',
        dest='threshold',
        type=float, help="Defines what is considered an 'improvement' in the score; if the score change is below this, it's treated as no improvement.")
    return parser

def parse_args(parser):
    args = parser.parse_args()
    if args.model_file:
        with open(args.model_file, 'r') as stream:
            model = yaml.safe_load(stream)

    if args.data_dir:
        data_dir = args.data_dir
    if args.output_dir:
        output_dir = args.output_dir
    ###
    if args.split_percentage is not None:
        split_percentage = args.split_percentage
    else:
        split_percentage = 0.8
    if args.learning_rate is not None:
        learning_rate = args.learning_rate
    if args.epochs is not None:
        epochs = args.epochs
    if args.batch_size is not None:
        batch_size = args.batch_size
    if args.kfold is not None:
        kfold = args.kfold
    else:
        kfold = None
    if args.savemodel is not None:
        savemodel = args.savemodel
    else:
        savemodel = False
    if args.seed is not None:
        seed = args.seed
    else:
        seed = 0
    ###
    if args.transfer_learning:
        transfer_learning = args.transfer_learning
    else:
        transfer_learning = None
    
    # Check if early stopping is enabled and other parameters are defined
    if args.early_stopping:
        if args.min_epochs is None or args.patience is None or args.threshold is None:
            parser.error("--min-epochs, --patience, and --threshold are required when --early-stopping is enabled.")

    if args.show_config:
        print_config()
        exit()

    return dict([('model', model),
            ("data_dir", data_dir),
            ("output_dir", output_dir), 
            ("transfer_learning",transfer_learning),
            ("split", split_percentage), 
            ("learning_rate", learning_rate), 
            ("epochs", epochs), 
            ("batch_size", batch_size), 
            ("seed", seed), 
            ("savemodel", savemodel),
            ("group_similar", args.group_similar), 
            ("early_stopping", args.early_stopping), 
            ("early_stopping_params", [args.min_epochs, args.patience, args.threshold]), 
            ("kfold",kfold)])

/home/adnanzai/.cache/pypoetry/virtualenvs/monai-train-y-Jg17XS-py3.9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed=0
set_determinism(seed=seed)
train_loader, val_loader, train_ds, val_ds = load_data('../sample',
                                                       0.5, 1.0, 4,
                                                       batch_size=1, image_size=[128,128,128], roi_size=[48,48,48],
                                                       seed=seed, group_similar=False)

# Initialize a list to store the first 100 images
images = []
masks = []

# Iterate over the train_loader
for batch in train_loader:
    # Assuming each batch is a single image due to batch_size=1, get the image data
    image = batch["image"][0]  # remove batch dimension
    image = image[0, image.shape[1] // 2, :, :].cpu().numpy()  # Take the middle slice of the first channel
    images.append(image)

    mask = batch["label"][0]  # remove batch dimension
    mask = mask[0, mask.shape[1] // 2, :, :].cpu().numpy()  # Take the middle slice of the first channel
    masks.append(mask)
    
    # Stop once we have 100 images
    if len(images) >= 100:
        break

# Set up the 3x2 grid plot
fig, axes = plt.subplots(2, 2, figsize=(7, 7))
# Plot each image in the 3x2 grid
for i, ax in enumerate(axes.flatten()):
    ax.imshow(images[i], cmap="bone")
    ax.axis("on")

# Save the figure as a PNG
plt.tight_layout()
plt.savefig("grid_image.png")
plt.close(fig)

fig2, axes2 = plt.subplots(2, 2, figsize=(7, 7))
# Plot each image in the 3x2 grid
for i, ax in enumerate(axes2.flatten()):
    ax.imshow(masks[i], cmap="bone")
    ax.axis("on")

# Save the figure as a PNG
plt.tight_layout()
plt.savefig("grid_mask.png")
plt.close(fig2)

/home/adnanzai/.cache/pypoetry/virtualenvs/monai-train-y-Jg17XS-py3.9/lib/python3.9/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.croppad.dictionary CropForegroundd.__init__:allow_smaller: Current default value of argument `allow_smaller=True` has been deprecated since version 1.2. It will be changed to `allow_smaller=False` in version 1.5.
  warn_deprecated(argname, msg, warning_category)


All conditions met.
***********

{'image': '../sample/imagesTr/cropped filtered_Odad1_CDOFB_E185_female_control_homozygote_671f6a8121231dfb31e88c4b276d8a13_RH.nii.gz', 'label': '../sample/labelsTr/cropped filtered_Odad1_CDOFB_E185_female_control_homozygote_671f6a8121231dfb31e88c4b276d8a13_RH.nii.gz'}
{'image': '../sample/imagesTr/cropped filtered_Hoxb13_HOXDB_E185_male_control_homozygote_3b22b8b8c45e204e300a45f8c6a1a28b_LH.nii.gz', 'label': '../sample/labelsTr/cropped filtered_Hoxb13_HOXDB_E185_male_control_homozygote_3b22b8b8c45e204e300a45f8c6a1a28b_LH.nii.gz'}
{'image': '../sample/imagesTr/cropped filtered_Zfp414_ZFFFB_E185_female_experimental_homozygote_caa1a267f966579f6dc671d00fe2ab9d_RH.nii.gz', 'label': '../sample/labelsTr/cropped filtered_Zfp414_ZFFFB_E185_female_experimental_homozygote_caa1a267f966579f6dc671d00fe2ab9d_RH.nii.gz'}
{'image': '../sample/imagesTr/cropped filtered_Kcnj13_KCNHB_E185_female_experimental_homozygote_3a364a4b9fc32ffd3455d9bcb69646f1_RH.nii.gz', 'label': 

Loading dataset: 100%|████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 16.40it/s]


In [3]:
for batch in train_loader:
    print(batch['image'].squeeze().shape)
    break

torch.Size([128, 128, 128])


In [4]:
# Initialize a list to store the first 100 images
images = []
masks = []

# Iterate over the train_loader
for batch in val_loader:
    # Assuming each batch is a single image due to batch_size=1, get the image data
    image = batch["image"][0]  # remove batch dimension
    image = image[0, image.shape[1] // 2, :, :].cpu().numpy()  # Take the middle slice of the first channel
    images.append(image)

    mask = batch["label"][0]  # remove batch dimension
    mask = mask[0, mask.shape[1] // 2, :, :].cpu().numpy()  # Take the middle slice of the first channel
    masks.append(mask)
    
    # Stop once we have 100 images
    if len(images) >= 100:
        break

# Set up the 3x2 grid plot
fig, axes = plt.subplots(2, 1, figsize=(7, 7))
# Plot each image in the 3x2 grid
for i, ax in enumerate(axes.flatten()):
    ax.imshow(images[i], cmap="bone")
    ax.axis("on")

# Save the figure as a PNG
plt.tight_layout()
plt.savefig("val_grid_image.png")
plt.close(fig)

fig2, axes2 = plt.subplots(2, 1, figsize=(7, 7))
# Plot each image in the 3x2 grid
for i, ax in enumerate(axes2.flatten()):
    ax.imshow(masks[i], cmap="bone")
    ax.axis("on")

# Save the figure as a PNG
plt.tight_layout()
plt.savefig("val_grid_mask.png")
plt.close(fig2)